## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: August 9, 2025

# Purpose: This notebook holds the final project for DSC540 - Data Preparation. Milestones 1-3 are included. The final project will merge three datasets into a single SQL table. We will create 5 data visualizations with the final result.

In [ ]:
# Imports 
import pandas as pd
import numpy as np
import requests
import json

# Milestone 1 - Project Research

# Air Quality in the USA

## Project Subject Area
My project will focus on air quality in the USA. I am interested in exploring relationships between various datasets, such as connections between air quality, CO₂ emissions, demographics, and asthma.

---

## Data Sources

### Flat File Source
- **Link:** [EPA Air Quality Data](https://www.kaggle.com/datasets/thedevastator/epa-air-quality-data)  
- **Background:** Contains thousands of rows of air quality data at both the state and local level.

### API Source
- **Link:** [EIA Open Data](https://www.eia.gov/opendata/)  
- **Background:** Energy Information Administration API with data such as total energy consumption, biomass, and CO₂ emissions.

### Website Source
- **Link:** [CDC Asthma Data Explorer API](https://ephtracking.cdc.gov/DataExplorer/api)  
- **Background:** Provides historical asthma data by state and year (2021 and prior). For example, child asthma rates are represented as a table with two columns: `State` and `Asthma Rate`.

---

## Relationships Between the Data
- The **flat file** contains location data down to the county level and includes a `ReportYear` column.  
- Both the **API** and **website** provide yearly data, allowing `ReportYear` to serve as a key linking variable.  
- If a dataset only includes state-level data, we will use the `StateName` field from the flat file to match records.  
- EIA provides county-level data for certain datasets (e.g., mines and refiners), which supports finer granularity.

These connections make combining datasets relatively straightforward and should yield useful insights.

---

## Ethical Considerations
- Data is sourced from **open-source, free-to-use government agencies**, minimizing bias and ethical concerns.  
- The data comes from multiple organizations, including energy and health agencies.  
- Potential health implications exist if a link is established between human activity and asthma rates or air quality.  
  - Example: A high concentration of oil refineries in Alabama could correspond to increased asthma rates and poorer air quality.  
- Public awareness of air quality impacts is critical.

---

## Project Approach
- Correlate **air quality** with **asthma rates** and **demographic changes** (e.g., population growth).
- Clean and prepare each data source at different project milestones to ensure quality before integration.
- Use SQL to merge datasets for analysis.
- Challenges include:
  - Need for temporal analysis (comparing states or locations over multiple years).
  - Reliance on publicly available data, which may have gaps or oversimplifications.

**Quality Assurance Measures:**
- Implement testing and validation to confirm data relationships.
- Seek peer and professor feedback for accuracy and methodology.

# Milestone 2 - Flat File Data Preparation

In [5]:
# Load data
df = pd.read_csv("air_quality_by_state_county.csv")
print(df.head())
print(df.shape)

   index  MeasureId                                        MeasureName  \
0      0         83  Number of days with maximum 8-hour average ozo...   
1      1         83  Number of days with maximum 8-hour average ozo...   
2      2         83  Number of days with maximum 8-hour average ozo...   
3      3         83  Number of days with maximum 8-hour average ozo...   
4      4         83  Number of days with maximum 8-hour average ozo...   

  MeasureType StratificationLevel  StateFips StateName  CountyFips CountyName  \
0      Counts      State x County          1   Alabama        1051     Elmore   
1      Counts      State x County          1   Alabama        1073  Jefferson   
2      Counts      State x County          1   Alabama        1079   Lawrence   
3      Counts      State x County          1   Alabama        1089    Madison   
4      Counts      State x County          1   Alabama        1097     Mobile   

   ReportYear  Value      Unit  UnitName    DataOrigin  MonitorOnly 

In [6]:
# Convert Value to a consistent numeric format

# Converts Value column to numeric, handling errors by coercing them to NaN
# This is important for ensuring that the data can be used for numerical analysis
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# Drop duplicates

# If the dataset contains duplicate rows, drop them
# This is important to ensure that each row represents a unique observation
df = df.drop_duplicates()

# Make new Column for county and state together

# There are some instances where Counties can match multiple states, so we're going to create unique identifiers
df['StateAndCounty'] = df['CountyName'].str.strip() + '_' + df['StateName'].str.strip()
# After this, we don't need the county name, so we'll drop it
df = df.drop(columns=['CountyName'])

# Drop Columns that are not needed

# The reason I'm dropping fips is because each simple corresponds to a location id. For example, every row with "Alabama" will have the same fips code. This is basically an encoded variable. Because we're already encoding our own variables, we don't need fips
df = df.drop(columns=['StateFips', 'CountyFips'])

# Dropping index because it isn't needed, it's going to present in a DF anyway
df = df.drop(columns=['index'])

# Dropping MeasureId since it's just the encoded version of the MeasureName
df = df.drop(columns=['MeasureId'])

print(df.columns)

# Standardize column names

# List of categorical columns to convert to numeric codes
cols_to_encode = [
    'MeasureName', 'MeasureType', 'StratificationLevel', 'StateName',
    'Unit', 'UnitName', 'DataOrigin', 'MonitorOnly','StateAndCounty'
]

# Loop through each column and apply factorization
for col in cols_to_encode:
    # pd.factorize converts each unique category to a unique integer
    # It returns a tuple: (array of codes, array of unique values)
    # Only the codes are assigned back to the DataFrame
    df[col], _ = pd.factorize(df[col])

# Display the first 5 rows of the modified DataFrame
print(df.head())

# Print the shape of the DataFrame (rows, columns)
print(df.shape)

flat_file_df_final = df.copy(deep=False)

Index(['MeasureName', 'MeasureType', 'StratificationLevel', 'StateName',
       'ReportYear', 'Value', 'Unit', 'UnitName', 'DataOrigin', 'MonitorOnly',
       'StateAndCounty'],
      dtype='object')
   MeasureName  MeasureType  StratificationLevel  StateName  ReportYear  \
0            0            0                    0          0        1999   
1            0            0                    0          0        1999   
2            0            0                    0          0        1999   
3            0            0                    0          0        1999   
4            0            0                    0          0        1999   

   Value  Unit  UnitName  DataOrigin  MonitorOnly  StateAndCounty  
0    5.0     0         0           0            0               0  
1   39.0     0         0           0            0               1  
2   28.0     0         0           0            0               2  
3   31.0     0         0           0            0               3  
4   32.0 

My data comprised numerical and categorical data, but most of my data is categories. Therefore, one of my top priorities was encoding data. I soon discovered that some columns such as 'Fips' were basically encoded names of other columns. Thefore, I dropped the State and County fip columns. The same applies to MeasureId. Index was also irrelevant so I dropped it as well. County wasn't a useful column on its own, so I combined it with State to make a new column so the StateAndCounty is unique, to avoid situations where different states had the same county names. Some rows might be duplicates, so I dropped those as well to avoid situations where data counted twice for certain cateogries. Finally, I converted values to a consistent numeric format.

With my changes, I saw 0 data loss, with 218,635 rows at the beginning of the data transformation and the same number at the end. The number of columns dropped from 15 to 11. These data transofmrations were neither risky nor carried any ethical complications. Simply put, the data is cleaner with less redundancy. As mentioned in milestone 1, my data comes from kaggle but their source is the EPA. 

I've achieved my goals with this data transformations, but I might change the transformations when dealing with the other sources (api and website). 

## Milestone 3 - Website/HTML Data Preparation

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_United_States_presidential_election_results_by_state"
tables = pd.read_html(url)

# Select the first table from the list of HTML tables
df = tables[0]

northeast_df = tables[1]
south_df = tables[2]
midwest_df = tables[3]
west_df = tables[4]


# Print the first 10 column names to understand the structure
print(df.columns)

# Preview the first 6 rows and last 6 columns using integer-based indexing
print(df.iloc[:10, -10:])

# Simple legend - R for republican, D for democrat

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
       36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
       54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67],
      dtype='int64')
     58      59  60    61    62    63      64    65    66           67
0  1996  2000 ‡ NaN  2004  2008  2012  2016 ‡  2020  2024        State
1     R       R NaN     R     R     R       R     R     R      Alabama
2     R       R NaN     R     R     R       R     R     R       Alaska
3     D       R NaN     R     R     R       R     D     R      Arizona
4     D       R NaN     R     R     R       R     R     R     Arkansas
5     D       D NaN     D     D     D       D     D     D   California
6     R       R NaN     R     D     D       D     D     D     Colorado
7     D       D NaN     D     D     D       D     D     D  Connecticut
8     D       D NaN     D     D     D  

## Transformation 1 - Add region to dataframe

In [ ]:
northeast_df[0]

0             State
1       Connecticut
2             Maine
3     Massachusetts
4     New Hampshire
          ...      
94              NaN
95              NaN
96              NaN
97              NaN
98              NaN
Name: 0, Length: 99, dtype: object

In [ ]:
# Create a mapping from state name to region
region_map = {}

for state in northeast_states:
    if state != "State":
        region_map[state] = "Northeast"
for state in south_states:
    if state != "State":
        region_map[state] = "South"
for state in midwest_states:
    if state != "State":
        region_map[state] = "Midwest"
for state in west_states:
    if state != "State":
        region_map[state] = "West"

# Determine the index for the new column (append to the end)
new_col_index = len(df.columns)

# Create the Region column values
region_column = ["Region"]  # Header for the new column
region_column += df.loc[1:, 0].map(region_map).tolist()  # Mapped regions for the rest

# Assign the new column to the DataFrame
df[new_col_index] = region_column

# Preview the result
print(df.iloc[:6, -6:])  # Show last 6 columns of the first 6 rows


     63      64    65    66          67      68
0  2012  2016 ‡  2020  2024       State  Region
1     R       R     R     R     Alabama   South
2     R       R     R     R      Alaska    West
3     R       R     D     R     Arizona    West
4     R       R     R     R    Arkansas   South
5     D       D     D     D  California    West


## Transformation 2 - Remove redundant columns

In [ ]:
# Drop the first state column (column 0)
df = df.drop(columns=[0])

# Drop columns where all values are NaN
df = df.dropna(axis=1, how='all')

print(df.head())

     1     2     3       4     6     7     8     9     10       11  ...    58  \
0  1790  1792  1796  1800 †  1804  1808  1812  1816  1820   1824 †  ...  1996   
1   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN    DR  Jackson  ...     R   
2   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     R   
3   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   
4   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   

       59    61    62    63      64    65    66        67      68  
0  2000 ‡  2004  2008  2012  2016 ‡  2020  2024     State  Region  
1       R     R     R     R       R     R     R   Alabama   South  
2       R     R     R     R       R     R     R    Alaska    West  
3       R     R     R     R       R     D     R   Arizona    West  
4       R     R     R     R       R     R     R  Arkansas   South  

[5 rows x 62 columns]


## Transformation 3 - Rename columns which are not state/region to make consistent naming 

In [ ]:
def clean_year(val):
    if isinstance(val, str):
        # Remove non-digit characters, keep only numbers
        digits = ''.join(filter(str.isdigit, val))
        return int(digits) if digits else val
    return val

df.iloc[0] = [
    clean_year(x) if x not in ("State", "Region") else x
    for x in df.iloc[0]
]
print(df.head())

     1     2     3     4     6     7     8     9     10       11  ...    58  \
0  1790  1792  1796  1800  1804  1808  1812  1816  1820     1824  ...  1996   
1   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN    DR  Jackson  ...     R   
2   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     R   
3   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   
4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   

     59    61    62    63    64    65    66        67      68  
0  2000  2004  2008  2012  2016  2020  2024     State  Region  
1     R     R     R     R     R     R     R   Alabama   South  
2     R     R     R     R     R     R     R    Alaska    West  
3     R     R     R     R     R     D     R   Arizona    West  
4     R     R     R     R     R     R     R  Arkansas   South  

[5 rows x 62 columns]


## Transformation 4 - Turn the indexes into columns

In [ ]:
# Set the first row as the header and remove it from the data
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

print(df.head())

0 1790 1792 1796 1800 1804 1808 1812 1816 1820     1824  ... 1996 2000 2004  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   DR  Jackson  ...    R    R    R   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    R    R    R   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    R    R   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    R    R   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    D    D   

0 2008 2012 2016 2020 2024       State Region  
0    R    R    R    R    R     Alabama  South  
1    R    R    R    R    R      Alaska   West  
2    R    R    R    D    R     Arizona   West  
3    R    R    R    R    R    Arkansas  South  
4    D    D    D    D    D  California   West  

[5 rows x 62 columns]


## Transformation 5 - Add missing data for the years

In [ ]:


# Identify integer-labeled year columns
year_columns = [col for col in df.columns if isinstance(col, int)]
year_columns_sorted = sorted(year_columns)

# Determine full range of years
all_years = list(range(year_columns_sorted[0], year_columns_sorted[-1] + 1))

# Track new columns to add (as a dict of {year: Series})
new_cols = {}

for year in all_years:
    if year not in df.columns:
        prev_years = [y for y in year_columns_sorted if y < year]
        if prev_years:
            prev_year = max(prev_years)
            new_cols[year] = df[prev_year]
        else:
            new_cols[year] = np.nan

# Combine new columns with the original dataframe
if new_cols:
    # Convert dict to DataFrame
    new_cols_df = pd.DataFrame(new_cols, index=df.index)

    # Merge the new year columns with the original
    df = pd.concat([df, new_cols_df], axis=1)

# Reorder columns: all years (sorted), then any non-year columns
year_cols_final = sorted([col for col in df.columns if isinstance(col, int)])
other_cols = [col for col in df.columns if not isinstance(col, int)]
df = df[year_cols_final + other_cols]

# (Optional) Defragment memory if many inserts were used before
df = df.copy()

# Preview
print(df.head())


  1790 1791 1792 1793 1794 1795 1796 1797 1798 1799  ... 2017 2018 2019 2020  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    D   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    D    D    D    D   

  2021 2022 2023 2024       State Region  
0    R    R    R    R     Alabama  South  
1    R    R    R    R      Alaska   West  
2    D    D    D    R     Arizona   West  
3    R    R    R    R    Arkansas  South  
4    D    D    D    D  California   West  

[5 rows x 237 columns]
